# Chicago Rideshare 2025: Pre-Trip Fare Prediction

Predicts the fare of a rideshare trip **before it starts**, from a pickup location, a destination and a date and time – like the price quote an app shows before you book.

**What the model can and cannot use**

| Known before the trip (used) | Known only after the trip (not used) |
|---|---|
| Pickup and destination community area | Actual trip minutes and miles |
| Date, hour, weekday, month, holiday | Actual route taken |
| Weather for the day | Tip and final charges |
| How busy the pickup area was in the **previous** hour | |
| Typical distance and duration of past trips between the same two areas | |

Because the actual minutes and miles are unknown, the model **estimates them from history**: what trips between the same two areas usually took.

**Before running:** run the main notebook (`Chicago_Rideshare_Surge_Analysis.ipynb`) first. It saves the cleaned data file this notebook starts from (`Data/rideshare_2025_clean.parquet`).

**Contents**
1. Setup and settings
2. Load the cleaned trips
3. Split by time: train, validate, test
4. Build pre-trip features
5. Train three models
6. Test the models on unseen months
7. What drives the prediction
8. Fare quote: enter a pickup location and destination
9. Conclusion (generated from the results)
10. References

**Packages:** `pandas`, `pyarrow`, `numpy`, `scikit-learn`, `matplotlib`, `joblib` (all included with Anaconda). `ipywidgets` is optional, for the dropdown menus in Section 8.

## 1. Setup and settings
**What this does:** imports the libraries and defines every path and setting in one place. Every number that affects the results has a comment explaining where it comes from. Numbers used only for chart layout (figure sizes, label spacing) change how charts look, never the results.

In [ ]:
import json
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import Markdown, display
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths (same project folder as the main notebook) ─────────────────────────
BASE_DIR = Path(os.environ.get("RIDESHARE_BASE_DIR", "/Users/williamdang/Documents/Project"))
CLEAN_PATH    = BASE_DIR / "Data" / "rideshare_2025_clean.parquet"            # written by the main notebook
GEOJSON_CACHE = BASE_DIR / "Data" / "chicago_community_areas.geojson"         # written by the main notebook
MODEL_PATH    = BASE_DIR / "Output" / "fare_quote_model.joblib"

YEAR = 2025                   # the data covers calendar year 2025
N_AREAS = 77                  # Chicago's 77 community areas, numbered 1-77

# ── Time-based split ─────────────────────────────────────────────────────────
TRAIN_END = f"{YEAR}-09-30"   # train on January-September (9 months: every season except late fall)
VALID_END = f"{YEAR}-10-31"   # validate on October (1 month, used to choose the best model)
# test on November-December (2 months the models never see, including Thanksgiving and the holidays)

# ── Sample sizes ─────────────────────────────────────────────────────────────
TRAIN_ROWS = 2_000_000
# Trips used to fit the models. Gradient boosting gains very little beyond a few million rows, and 2M keeps
# training to a few minutes on a laptop. The route history (Section 4.3) still uses every training trip.
EVAL_ROWS = 1_000_000         # trips used to score validation and test: enough for errors precise to about 1 cent
IMPORTANCE_ROWS = 50_000      # trips used for permutation importance (slow method; 50k is plenty for a ranking)
RANDOM_SEED = 42              # fixed seed so the same samples are drawn every run (42 is just a convention)

# ── Cleaning ─────────────────────────────────────────────────────────────────
FARE_MAX_PCTL = 99.9
# Drop the top 0.1% of fares (computed on training months). These are mostly recording errors (e.g. a
# 1,000-dollar fare for a short trip) and would pull a squared-error model toward impossible values.

# ── Route history ────────────────────────────────────────────────────────────
MIN_ROUTE_TRIPS = 20
# A pickup -> destination pair needs at least 20 training trips before its typical miles/minutes are trusted.
# With 20 trips the median is stable to within roughly 10%; rarer pairs fall back to straight-line distance.

# ── Recent demand ────────────────────────────────────────────────────────────
DEMAND_BASELINE_WEEKS = 4
# "Normal" previous-hour demand = average of the same area, same hour, same weekday over the past 4 weeks
# (past only, so it is known at booking time). Matches the benchmark used in the main notebook.
HOURS_PER_WEEK = 24 * 7       # 168: one week back is the same weekday and hour

# ── Gradient boosting ────────────────────────────────────────────────────────
GB_PARAMS = dict(
    learning_rate=0.1,        # scikit-learn's default step size: a good balance of speed and accuracy
    max_iter=500,             # upper limit on trees; early stopping usually ends well before this
    max_leaf_nodes=63,        # allows detailed interactions (area x hour x distance); default 31 is coarser
    min_samples_leaf=100,     # each leaf averages at least 100 trips, which smooths the 2.50-dollar fare rounding
    l2_regularization=1.0,    # mild penalty against over-fitting rare combinations
    early_stopping=True,      # stop adding trees when a held-out 10% of training data stops improving
    validation_fraction=0.1,
    n_iter_no_change=20,      # 20 rounds without improvement before stopping
    random_state=RANDOM_SEED,
)

# ── Fare data facts (References [1], [2]) ────────────────────────────────────
FARE_ROUNDING = 2.50          # the data rounds every fare to the nearest 2.50 dollars
ROUNDING_MAE_FLOOR = FARE_ROUNDING / 4
# Even a perfect model is off by the rounding: a fare rounded to the nearest 2.50 is off by between 0 and
# 1.25 dollars, 0.625 on average (= 2.50 / 4). No model can score a lower average error than this.
TOLERANCE = FARE_ROUNDING     # "accurate" = predicted within one rounding step (2.50 dollars) of the actual fare

TIME_CATEGORIES = ["Morning", "Noon", "Evening", "Night"]
CATEGORY_HOURS  = {"Morning": "6–10 AM", "Noon": "10 AM–4 PM", "Evening": "4–9 PM", "Night": "9 PM–6 AM"}

## 2. Load the cleaned trips
**What this does:** reads only the columns needed from the cleaned file and keeps trips that can be quoted: both the pickup and the destination community area are known, and fare, miles and minutes are positive.

**Why both areas are required:** the quote is built from "where from" and "where to". A trip without a destination area (usually one that ends outside Chicago) has no route to learn from.

In [ ]:
assert CLEAN_PATH.exists(), f"{CLEAN_PATH} not found - run the main notebook first (Section 6.3 saves it)."

cols = [
    "trip_start_timestamp", "Pickup_Date", "Pickup_Hour", "Pickup_Day_of_Week", "Pickup_Month",
    "Pickup_Time_Category", "pickup_community_area", "dropoff_community_area",
    "fare", "trip_miles", "Trip_Minutes", "PRCP (Inches)", "SNOW (Inches)",
]
trips_all = pd.read_parquet(CLEAN_PATH, columns=cols)
print(f"Trips in the cleaned file: {len(trips_all):,}")

usable = (
    trips_all["pickup_community_area"].notna()
    & trips_all["dropoff_community_area"].notna()
    & (trips_all["fare"] > 0)
    & (trips_all["trip_miles"] > 0)
    & (trips_all["Trip_Minutes"] > 0)
)
trips = trips_all.loc[usable].reset_index(drop=True)
trips["pickup_community_area"] = trips["pickup_community_area"].astype("int16")
trips["dropoff_community_area"] = trips["dropoff_community_area"].astype("int16")
print(f"Trips with a pickup and destination area and a valid fare: {len(trips):,} ({usable.mean() * 100:.1f}%)")

## 3. Split by time: train, validate, test
**What this does:** splits the year into three blocks:
- **Train – January to September:** the models learn from these trips.
- **Validate – October:** used to compare the models and pick the best one.
- **Test – November and December:** scored once at the end, never used for any choice.

**Why split by time instead of at random:** a real quote engine predicts **future** trips from **past** ones. A random split would let the model learn from trips on the same day and hour it is tested on, which makes it look better than it will be in real use. Testing on later months shows how it holds up when the calendar moves on.

**Why drop the top 0.1% of fares:** those are mostly recording errors. The cutoff is computed on the training months only, so the test months do not influence it.

In [ ]:
date = trips["Pickup_Date"]
split = np.select(
    [date <= pd.Timestamp(TRAIN_END), date <= pd.Timestamp(VALID_END)],
    ["train", "valid"],
    default="test",
)
trips["Split"] = pd.Categorical(split, ["train", "valid", "test"])

fare_cap = float(trips.loc[trips["Split"] == "train", "fare"].quantile(FARE_MAX_PCTL / 100))
too_high = trips["fare"] > fare_cap
print(f"Fare cap ({FARE_MAX_PCTL}th percentile of training fares): {fare_cap:,.2f} dollars "
      f"-> {int(too_high.sum()):,} trips removed ({too_high.mean() * 100:.2f}%)")
trips = trips.loc[~too_high].reset_index(drop=True)

display(trips["Split"].value_counts(sort=False).to_frame("Trips"))

## 4. Build pre-trip features
Every feature below is something an app would know **at the moment of booking**.

### 4.1 Calendar features
**What this does:** hour, weekday, month, weekend, US federal holiday and time-of-day category.

**Why:** prices and traffic follow daily and weekly rhythms (rush hours, weekend nights), and holidays behave unlike ordinary days. Holidays come from pandas' US federal holiday calendar [4].

In [ ]:
holidays = USFederalHolidayCalendar().holidays(start=f"{YEAR}-01-01", end=f"{YEAR}-12-31")

trips["hour"] = trips["Pickup_Hour"].astype("int8")
trips["weekday"] = trips["Pickup_Day_of_Week"].cat.codes.astype("int8")      # 0 = Monday ... 6 = Sunday
trips["month"] = trips["Pickup_Month"].cat.codes.astype("int8") + 1          # 1 = January ... 12 = December
trips["is_weekend"] = (trips["weekday"] >= 5).astype("int8")                 # 5, 6 = Saturday, Sunday
trips["is_holiday"] = trips["Pickup_Date"].isin(holidays).astype("int8")
trips["time_category"] = trips["Pickup_Time_Category"].cat.codes.astype("int8")
trips["rain_in"] = trips["PRCP (Inches)"].astype("float32")
trips["snow_in"] = trips["SNOW (Inches)"].astype("float32")

print("Holidays in the data:", ", ".join(d.strftime("%b %d") for d in holidays))

### 4.2 Straight-line distance between the two areas
**What this does:** finds the center point of every community area from the boundary file and measures the straight-line ("as the crow flies") distance between the pickup and destination centers.

**Why:** distance is the biggest part of any fare. Straight-line distance is always available, even for a pair of areas that rarely appears in the data. It underestimates the real driving distance, so the model learns the typical road-to-straight-line ratio from the data.

In [ ]:
geo = json.loads(GEOJSON_CACHE.read_text())
EARTH_RADIUS_MILES = 3958.8          # mean radius of the Earth, used by the haversine distance formula

centers = np.full((N_AREAS + 1, 2), np.nan)                     # row = area number, columns = (lat, lon)
area_names = {}
for f in geo["features"]:
    props, g = f["properties"], f["geometry"]
    area = int(props.get("area_numbe", props.get("Community_Code")))
    name = str(props.get("community", props.get("Community_Name"))).title()
    rings = g["coordinates"] if g["type"] == "Polygon" else [r for poly in g["coordinates"] for r in poly]
    pts = np.array([p for ring in rings for p in ring])          # every boundary point (lon, lat)
    centers[area] = [pts[:, 1].mean(), pts[:, 0].mean()]         # average of the boundary points ~ center
    area_names[area] = name


def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_MILES * np.arcsin(np.sqrt(a))


# 78 x 78 table: straight-line miles between every pair of area centers (row/column = area number)
straight_miles = haversine_miles(centers[:, None, 0], centers[:, None, 1], centers[None, :, 0], centers[None, :, 1])

pu = trips["pickup_community_area"].to_numpy()
do = trips["dropoff_community_area"].to_numpy()
trips["straight_line_miles"] = straight_miles[pu, do].astype("float32")
print(f"Area centers found: {np.isfinite(centers[1:, 0]).sum()} of {N_AREAS}")
display(trips["straight_line_miles"].describe().round(2))

### 4.3 Route history: typical miles and minutes between the two areas
**What this does:** for every pickup → destination pair, the **median** miles and minutes of past trips, overall and for each time of day (a 5 PM crosstown trip takes longer than the same trip at 5 AM).

**Why:** this is the model's best guess of the trip it is quoting, standing in for the actual miles and minutes it cannot know yet. The median is used instead of the average so a few odd trips (a detour, a wrong recording) do not distort it.

**Leakage guard:** route history is built from **training months only**. If it used October–December trips, the model would be peeking at the trips it is later tested on.

**Rare pairs:** a pair needs at least 20 training trips (`MIN_ROUTE_TRIPS`) to have a trusted history. Rarer pairs are left blank and the model relies on straight-line distance instead.

In [ ]:
train_trips = trips[trips["Split"] == "train"]
keys = ["pickup_community_area", "dropoff_community_area"]

route = train_trips.groupby(keys).agg(
    route_trips=("fare", "size"),
    route_median_miles=("trip_miles", "median"),
    route_median_minutes=("Trip_Minutes", "median"),
)
route = route[route["route_trips"] >= MIN_ROUTE_TRIPS]

route_time = (
    train_trips.groupby(keys + ["time_category"])
    .agg(n=("fare", "size"), route_time_median_minutes=("Trip_Minutes", "median"))
)
route_time = route_time.loc[route_time["n"] >= MIN_ROUTE_TRIPS, ["route_time_median_minutes"]]

trips = trips.merge(route, left_on=keys, right_index=True, how="left")
trips = trips.merge(route_time, left_on=keys + ["time_category"], right_index=True, how="left")
trips["route_trips"] = np.log1p(trips["route_trips"].fillna(0))     # log scale: 20 vs 200 trips matters more than 20,000 vs 20,200

# Typical road-miles per straight-line mile (training months), used to fill rare pairs for the rate-card model
road_factor = float((train_trips["trip_miles"] / train_trips["straight_line_miles"].where(train_trips["straight_line_miles"] > 0.5)).median())
minutes_per_mile = float((train_trips["Trip_Minutes"] / train_trips["trip_miles"]).median())

coverage = trips["route_median_miles"].notna().groupby(trips["Split"], observed=True).mean() * 100
print(f"Pickup -> destination pairs with a trusted history: {len(route):,}")
print(f"Road miles per straight-line mile (median): {road_factor:.2f} | minutes per mile (median): {minutes_per_mile:.2f}")
print("Share of trips whose route has a history (%):")
display(coverage.round(1).to_frame("Covered_%"))

### 4.4 Recent demand in the pickup area
**What this does:** two numbers for the pickup area:
- **Pickups in the previous hour** – how busy the area is right now.
- **Previous-hour pickups ÷ normal for that hour** – the same 1.0 = normal, 1.3 = 30% busier idea as the main notebook. "Normal" is the average of the same area, same weekday and hour over the past 4 weeks (`DEMAND_BASELINE_WEEKS`).

**Why the previous hour:** the current hour is not over when a trip is booked, so using it would leak information the app does not have yet. The previous hour is complete and known.

**Note:** the main notebook found surge-style premiums depend on how busy an area is relative to normal; this feature lets the quote reflect that.

In [ ]:
dates = pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31")
N_HOURS = len(dates) * 24
pickups = trips_all.loc[trips_all["pickup_community_area"].notna()]          # all pickups, with or without a destination
p_area = pickups["pickup_community_area"].astype("int64").to_numpy()
p_hour = ((pickups["Pickup_Date"] - dates[0]).dt.days.to_numpy() * 24 + pickups["Pickup_Hour"].to_numpy()).astype("int64")
ok = (p_area >= 1) & (p_area <= N_AREAS) & (p_hour >= 0) & (p_hour < N_HOURS)
grid = np.bincount((p_area[ok] - 1) * N_HOURS + p_hour[ok], minlength=N_AREAS * N_HOURS).reshape(N_AREAS, N_HOURS).astype("float64")


def lag(a, hours):
    """Shift the hourly grid so each hour sees the value from `hours` earlier (blank at the start of the year)."""
    out = np.full_like(a, np.nan)
    out[:, hours:] = a[:, :-hours]
    return out


prev_hour = lag(grid, 1)
with np.errstate(invalid="ignore", divide="ignore"), warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # the first 4 weeks of the year have no history yet -> blank, not an error
    normal_prev = np.nanmean(np.stack([lag(grid, 1 + HOURS_PER_WEEK * k) for k in range(1, DEMAND_BASELINE_WEEKS + 1)]), axis=0)
    demand_ratio = np.where(normal_prev > 0, prev_hour / normal_prev, np.nan)

t_idx = ((trips["Pickup_Date"] - dates[0]).dt.days.to_numpy() * 24 + trips["hour"].to_numpy()).astype("int64")
a_idx = trips["pickup_community_area"].to_numpy().astype("int64") - 1
trips["prev_hour_pickups"] = prev_hour[a_idx, t_idx].astype("float32")
trips["demand_ratio_prev_hour"] = demand_ratio[a_idx, t_idx].astype("float32")

display(trips[["prev_hour_pickups", "demand_ratio_prev_hour"]].describe(percentiles=[.1, .5, .9]).round(2))

## 5. Train three models
Three models of increasing detail, so it is clear how much each step adds:

| Model | How it predicts | Why include it |
|---|---|---|
| **A. Route median** | The median fare of past trips between the same two areas | The simplest sensible quote; any useful model must beat it |
| **B. Rate card** | Base + per-minute + per-mile, applied to the route's typical minutes and miles | The fare recipe from the main notebook, using estimated instead of actual trip length |
| **C. Gradient boosting** | Many small decision trees, each correcting the last, using every feature | Learns interactions such as "airport trips at 5 PM in snow" [3] |

**Why gradient boosting:** it handles a mix of numbers and categories (area codes), copes with blank values on its own (e.g. a rare route), and captures non-straight-line effects without hand-written rules. scikit-learn's histogram version is fast on millions of rows [3].

In [ ]:
NUMERIC_FEATURES = [
    "hour", "weekday", "month", "is_weekend", "is_holiday", "rain_in", "snow_in",
    "straight_line_miles", "route_median_miles", "route_median_minutes", "route_time_median_minutes", "route_trips",
    "prev_hour_pickups", "demand_ratio_prev_hour",
]
CATEGORICAL_FEATURES = ["pickup_community_area", "dropoff_community_area", "time_category"]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

rng = np.random.default_rng(RANDOM_SEED)


def sample(split_name, n):
    idx = np.flatnonzero((trips["Split"] == split_name).to_numpy())
    return trips.iloc[np.sort(rng.choice(idx, size=min(n, len(idx)), replace=False))]


train = sample("train", TRAIN_ROWS)
valid = sample("valid", EVAL_ROWS)
test = sample("test", EVAL_ROWS)
print(f"Training sample: {len(train):,} | validation: {len(valid):,} | test: {len(test):,}")

# ── Model A: route median fare (fallbacks: pickup-area median, then citywide median) ──
route_fare = train_trips.groupby(keys)["fare"].agg(["median", "size"])
route_fare = route_fare.loc[route_fare["size"] >= MIN_ROUTE_TRIPS, "median"]
pickup_fare = train_trips.groupby("pickup_community_area")["fare"].median()
city_fare = float(train_trips["fare"].median())


def predict_route_median(frame):
    pred = frame.join(route_fare.rename("p"), on=keys)["p"]
    pred = pred.fillna(frame["pickup_community_area"].map(pickup_fare))
    return pred.fillna(city_fare).to_numpy()


# ── Model B: rate card on the route's typical miles and minutes ──
def rate_card_inputs(frame):
    miles = frame["route_median_miles"].fillna(frame["straight_line_miles"] * road_factor)
    minutes = frame["route_time_median_minutes"].fillna(frame["route_median_minutes"]).fillna(miles * minutes_per_mile)
    return pd.DataFrame({"expected_minutes": minutes, "expected_miles": miles})


rate_card = LinearRegression().fit(rate_card_inputs(train), train["fare"])
print(f"Rate card on expected trips: base {rate_card.intercept_:.2f} + {rate_card.coef_[0]:.3f}/min "
      f"+ {rate_card.coef_[1]:.3f}/mile (dollars)")

# ── Model C: gradient boosting on every pre-trip feature ──
gb = HistGradientBoostingRegressor(categorical_features=[FEATURES.index(c) for c in CATEGORICAL_FEATURES], **GB_PARAMS)
gb.fit(train[FEATURES], train["fare"])
if gb.n_iter_ < GB_PARAMS["max_iter"]:
    print(f"Gradient boosting stopped early after {gb.n_iter_} trees (more trees no longer helped)")
else:
    print(f"Gradient boosting used the full {gb.n_iter_} trees; raising max_iter in Section 1 may add a little accuracy")

MODELS = {
    "A. Route median": predict_route_median,
    "B. Rate card": lambda f: rate_card.predict(rate_card_inputs(f)),
    "C. Gradient boosting": lambda f: gb.predict(f[FEATURES]),
}

## 6. Test the models on unseen months
**How the models are scored**

| Measure | Meaning |
|---|---|
| **MAE** | Average dollars the quote is off by. The easiest to read |
| **RMSE** | Like MAE but punishes big misses more |
| **MAPE** | Average % off, so a 2-dollar miss on a 10-dollar ride counts more than on a 50-dollar ride |
| **R²** | Share of fare differences between trips the model explains (1 = all) |
| **Within 2.50** | Share of quotes within one 2.50-dollar rounding step of the actual fare |

**Best possible score:** fares are rounded to the nearest 2.50 dollars [1], so even a perfect model is off by 0.625 dollars on average (`ROUNDING_MAE_FLOOR`). Judge MAE against that floor, not against zero.

October is used to pick the best model; November–December give the final, untouched score.

In [ ]:
def score(y, p):
    err = p - y
    return {
        "MAE": np.mean(np.abs(err)),
        "RMSE": np.sqrt(np.mean(err ** 2)),
        "MAPE_%": np.mean(np.abs(err) / y) * 100,
        "R2": 1 - np.sum(err ** 2) / np.sum((y - y.mean()) ** 2),
        f"Within_{TOLERANCE:.2f}_%": np.mean(np.abs(err) <= TOLERANCE) * 100,
    }


rows = []
preds = {}
for split_name, frame in [("Validation (Oct)", valid), ("Test (Nov-Dec)", test)]:
    y = frame["fare"].to_numpy()
    for name, fn in MODELS.items():
        p = fn(frame)
        preds[(split_name, name)] = p
        rows.append({"Data": split_name, "Model": name, **score(y, p)})
scores = pd.DataFrame(rows).set_index(["Data", "Model"])

best_model = scores.loc["Validation (Oct)", "MAE"].idxmin()           # chosen on October only
print(f"Best model on validation (lowest MAE): {best_model}  |  rounding floor for MAE: {ROUNDING_MAE_FLOOR:.3f} dollars")
display(scores.round(3))

### 6.1 Where the best model is accurate and where it struggles
**What this does:** compares predicted and actual fares on the test months and breaks the error down by time of day and by fare size.

In [ ]:
y_test = test["fare"].to_numpy()
p_best = preds[("Test (Nov-Dec)", best_model)]
abs_err = np.abs(p_best - y_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs actual (hexbin: darker = more trips)
lim = np.percentile(y_test, 99)                                   # zoom to the 99% of fares where most trips are
hb = axes[0].hexbin(y_test, p_best, gridsize=60, extent=(0, lim, 0, lim), bins="log", cmap="Blues", mincnt=1)
axes[0].plot([0, lim], [0, lim], color="black", linewidth=1)
axes[0].set_xlabel("Actual fare (dollars)")
axes[0].set_ylabel("Predicted fare (dollars)")
axes[0].set_title(f"{best_model}: predicted vs. actual (test)", fontweight="bold")
fig.colorbar(hb, ax=axes[0], label="Trips (log scale)")

# MAE by time of day, every model
cat_codes = test["time_category"].to_numpy()
width = 0.25                                                     # bar width for 3 models side by side
for i, name in enumerate(MODELS):
    err = np.abs(preds[("Test (Nov-Dec)", name)] - y_test)
    vals = [err[cat_codes == k].mean() for k in range(len(TIME_CATEGORIES))]
    axes[1].bar(np.arange(len(TIME_CATEGORIES)) + (i - 1) * width, vals, width, label=name)
axes[1].axhline(ROUNDING_MAE_FLOOR, color="black", linestyle="--", linewidth=1, label="Rounding floor")
axes[1].set_xticks(range(len(TIME_CATEGORIES)))
axes[1].set_xticklabels([f"{c}\n{CATEGORY_HOURS[c]}" for c in TIME_CATEGORIES])
axes[1].set_ylabel("Average error (dollars)")
axes[1].set_title("Average error by time of day (test)", fontweight="bold")
axes[1].legend(fontsize=9)

# MAE by fare size
FARE_BANDS = [0, 10, 20, 30, 50, np.inf]                          # dollar bands: short, medium, long, very long rides
band = pd.cut(y_test, FARE_BANDS, right=False)
band_mae = pd.Series(abs_err).groupby(band, observed=True).mean()
band_share = pd.Series(abs_err).groupby(band, observed=True).size() / len(abs_err) * 100
bars = axes[2].bar([str(b) for b in band_mae.index], band_mae.values, color="#3b6ea5")
axes[2].bar_label(bars, labels=[f"{s:.0f}% of trips" for s in band_share.values], padding=3, fontsize=9)
axes[2].set_xlabel("Actual fare band (dollars)")
axes[2].set_ylabel("Average error (dollars)")
axes[2].set_title(f"{best_model}: error by fare size (test)", fontweight="bold")

for ax in axes[1:]:
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

by_cat = pd.DataFrame({
    "Trips": pd.Series(cat_codes).value_counts().sort_index().to_numpy(),
    "MAE": [abs_err[cat_codes == k].mean() for k in range(len(TIME_CATEGORIES))],
    f"Within_{TOLERANCE:.2f}_%": [np.mean(abs_err[cat_codes == k] <= TOLERANCE) * 100 for k in range(len(TIME_CATEGORIES))],
}, index=TIME_CATEGORIES)
display(by_cat.round(3))

## 7. What drives the prediction
**What this does:** permutation importance [5] – shuffle one feature at a time across trips and measure how much worse the gradient-boosting model gets. A big increase in error means the model relies heavily on that feature.

**Why this method:** it measures importance on the test months, in dollars of extra error, and treats every feature the same way whether it is a number or a category.

In [ ]:
imp_sample = test.sample(n=min(IMPORTANCE_ROWS, len(test)), random_state=RANDOM_SEED)
imp = permutation_importance(
    gb, imp_sample[FEATURES], imp_sample["fare"],
    scoring="neg_mean_absolute_error",
    n_repeats=3,                 # shuffle each feature 3 times and average, to steady the estimate
    random_state=RANDOM_SEED,
)
importance = (
    pd.DataFrame({"Feature": FEATURES, "Extra_MAE_dollars": imp.importances_mean, "Std": imp.importances_std})
    .sort_values("Extra_MAE_dollars", ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance["Feature"], importance["Extra_MAE_dollars"], xerr=importance["Std"], color="#3b6ea5")
ax.set_xlabel("Increase in average error when the feature is shuffled (dollars)")
ax.set_title("What the fare quote relies on (test months)", fontweight="bold")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

display(importance.sort_values("Extra_MAE_dollars", ascending=False).round(3).reset_index(drop=True))

## 8. Fare quote: enter a pickup location and destination
**What this does:** `quote_fare()` turns a pickup location, a destination and a date and time into a predicted fare, using the best model from Section 6.

**How to enter locations:** the data records locations as Chicago's 77 community areas, so locations are entered as an area **name** (e.g. `"Loop"`, `"O'Hare"`, `"Lincoln Park"`) or **number** (1–77). Names ignore capital letters, spaces and punctuation (`"ohare"` works for O'Hare), and a partial name works if it matches only one area. Run `list_areas()` to see every name.

**Optional inputs:**
- `rain_in` and `snow_in` – expected inches for the day (default 0).
- `prev_hour_pickups` and `demand_ratio` – how busy the pickup area is right now. If left out, the model treats them as unknown and uses what is typical for that area and time.

The model and everything it needs are also saved to `Output/fare_quote_model.joblib`, so the quote can be reused without rerunning the notebook.

In [ ]:
def simplify(text):
    """Lower-case and drop spaces and punctuation, so "O'Hare", "ohare" and "o hare" all match."""
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


name_to_area = {simplify(name): area for area, name in area_names.items()}


def list_areas():
    """Show every community area number and name."""
    return pd.DataFrame(sorted(area_names.items()), columns=["Area_Number", "Community_Area"]).set_index("Area_Number")


def find_area(location):
    """Turn an area name (full or partial, any case) or number into an area number."""
    if isinstance(location, (int, np.integer)) or str(location).strip().isdigit():
        area = int(location)
        if area in area_names:
            return area
        raise ValueError(f"Area number must be 1-{N_AREAS}.")
    text = simplify(location)
    if text in name_to_area:
        return name_to_area[text]
    matches = [a for n, a in name_to_area.items() if text in n]
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise ValueError(f"No community area matches '{location}'. Run list_areas() to see the names.")
    raise ValueError(f"'{location}' matches several areas: {', '.join(area_names[a] for a in matches)}. Be more specific.")


def quote_fare(pickup, destination, when, rain_in=0.0, snow_in=0.0, prev_hour_pickups=None, demand_ratio=None):
    """Predicted fare (dollars) for a trip from `pickup` to `destination` starting at `when`."""
    pu_area, do_area = find_area(pickup), find_area(destination)
    t = pd.Timestamp(when)
    hour = t.hour
    cat = int(np.array([3]*6 + [0]*4 + [1]*6 + [2]*5 + [3]*3)[hour])   # same hour -> category map as the main notebook
    row = pd.DataFrame([{
        "pickup_community_area": pu_area, "dropoff_community_area": do_area, "time_category": cat,
        "hour": hour, "weekday": t.dayofweek, "month": t.month,
        "is_weekend": int(t.dayofweek >= 5), "is_holiday": int(t.normalize() in set(holidays)),
        "rain_in": rain_in, "snow_in": snow_in,
        "straight_line_miles": straight_miles[pu_area, do_area],
        "prev_hour_pickups": np.nan if prev_hour_pickups is None else prev_hour_pickups,
        "demand_ratio_prev_hour": np.nan if demand_ratio is None else demand_ratio,
    }])
    r = route.reindex(pd.MultiIndex.from_tuples([(pu_area, do_area)], names=keys))
    row["route_median_miles"] = r["route_median_miles"].to_numpy()
    row["route_median_minutes"] = r["route_median_minutes"].to_numpy()
    row["route_trips"] = np.log1p(r["route_trips"].fillna(0).to_numpy())
    rt = route_time.reindex(pd.MultiIndex.from_tuples([(pu_area, do_area, cat)], names=keys + ["time_category"]))
    row["route_time_median_minutes"] = rt["route_time_median_minutes"].to_numpy()

    fare = float(MODELS[best_model](row)[0])
    typical_error = float(scores.loc[("Test (Nov-Dec)", best_model), "MAE"])
    miles = row["route_median_miles"].iloc[0]
    miles_txt = f"{miles:.1f} typical road miles" if pd.notna(miles) else \
        f"{row['straight_line_miles'].iloc[0] * road_factor:.1f} estimated road miles (rare route)"
    return {
        "Pickup": area_names[pu_area], "Destination": area_names[do_area], "When": t.strftime("%a %b %d, %I:%M %p"),
        "Predicted_Fare": round(fare, 2), "Typical_Error": round(typical_error, 2),
        "Likely_Range": f"{max(fare - typical_error, 0):.2f} - {fare + typical_error:.2f} dollars",
        "Route": miles_txt,
    }


# Save everything the quote needs
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({
    "gradient_boosting": gb, "rate_card": rate_card, "features": FEATURES, "best_model": best_model,
    "route": route, "route_time": route_time, "route_fare": route_fare, "pickup_fare": pickup_fare, "city_fare": city_fare,
    "straight_miles": straight_miles, "area_names": area_names, "road_factor": road_factor, "minutes_per_mile": minutes_per_mile,
    "holidays": list(holidays), "test_scores": scores.loc["Test (Nov-Dec)"],
}, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

### 8.1 Try it
Change the pickup, destination and time below and rerun the cell. The first example uses the two busiest pickup areas in your data, so it always runs.

In [ ]:
busiest = trips["pickup_community_area"].value_counts().index[:2]
example_pickup, example_destination = area_names[int(busiest[0])], area_names[int(busiest[1])]

# ── Edit these ────────────────────────────────────────────────────────────────
pickup = example_pickup               # e.g. "Loop", "Near North Side", "O'Hare", or a number 1-77
destination = example_destination     # e.g. "Lincoln Park", "Hyde Park"
when = "2025-12-12 18:00"             # date and time of pickup
# ─────────────────────────────────────────────────────────────────────────────

display(pd.Series(quote_fare(pickup, destination, when)).to_frame("Quote"))

# The same trip at different times of day
times = ["2025-12-12 03:00", "2025-12-12 08:00", "2025-12-12 13:00", "2025-12-12 18:00", "2025-12-12 22:00"]
display(pd.DataFrame([quote_fare(pickup, destination, t) for t in times])[["When", "Predicted_Fare", "Likely_Range"]])

### 8.2 Pick from menus (optional)
**What this does:** dropdown menus for pickup and destination, a date-and-time box and a button. Needs `ipywidgets` (`pip install ipywidgets`); if it is not installed, use the cell above instead.

In [ ]:
try:
    import ipywidgets as widgets

    options = sorted((name, area) for area, name in area_names.items())
    pick = widgets.Dropdown(options=options, value=int(busiest[0]), description="Pickup:")
    dest = widgets.Dropdown(options=options, value=int(busiest[1]), description="Destination:")
    when_box = widgets.Text(value="2025-12-12 18:00", description="When:")
    button = widgets.Button(description="Get fare quote", button_style="primary")
    out = widgets.Output()

    def on_click(_):
        with out:
            out.clear_output()
            try:
                q = quote_fare(pick.value, dest.value, when_box.value)
                display(Markdown(f"**{q['Pickup']} → {q['Destination']}**, {q['When']}: about **{q['Predicted_Fare']:.2f} dollars** "
                                 f"(likely {q['Likely_Range']}; {q['Route']})"))
            except Exception as e:
                print("Could not quote:", e)

    button.on_click(on_click)
    display(widgets.VBox([pick, dest, when_box, button, out]))
except ImportError:
    print("ipywidgets is not installed - use the quote cell above, or run: pip install ipywidgets")

## 9. Conclusion
Generated from the results when the notebook runs.

In [ ]:
t = scores.loc["Test (Nov-Dec)"]
best, base, rc_ = t.loc[best_model], t.loc["A. Route median"], t.loc["B. Rate card"]
within_col = f"Within_{TOLERANCE:.2f}_%"
top3 = importance.sort_values("Extra_MAE_dollars", ascending=False)["Feature"].head(3).tolist()
worst_cat = by_cat["MAE"].idxmax()
best_cat = by_cat["MAE"].idxmin()
big_band = band_mae.index[-1]

lines = [
    f"**Best model: {best_model}** (chosen on October, scored on November–December, which it never saw).",
    f"**Accuracy.** On the test months the quote is off by **{best['MAE']:.2f} dollars on average** (RMSE {best['RMSE']:.2f}, "
    f"{best['MAPE_%']:.1f}% of the fare), and **{best[within_col]:.0f}% of quotes are within one 2.50-dollar step** of the actual fare. "
    f"It explains {best['R2'] * 100:.0f}% of the differences in fares between trips.",
    f"**Compared with simpler quotes.** The route-median quote is off by {base['MAE']:.2f} dollars and the rate card by {rc_['MAE']:.2f}; "
    f"the best model cuts the route-median error by **{(1 - best['MAE'] / base['MAE']) * 100:.0f}%**.",
    f"**How close to perfect.** Because fares are rounded to 2.50 dollars, no model can average less than "
    f"{ROUNDING_MAE_FLOOR:.3f} dollars of error; the best model is {best['MAE'] - ROUNDING_MAE_FLOOR:.2f} dollars above that floor.",
    f"**What drives the quote.** The features it relies on most are {', '.join(top3)} – essentially *how far* and *how long* "
    f"the trip between the two areas usually is.",
    f"**Time of day.** Quotes are most accurate for {best_cat} trips ({CATEGORY_HOURS[best_cat]}; {by_cat.loc[best_cat, 'MAE']:.2f} dollars "
    f"average error) and least accurate for {worst_cat} trips ({CATEGORY_HOURS[worst_cat]}; {by_cat.loc[worst_cat, 'MAE']:.2f} dollars).",
    f"**Long trips** (fares of {big_band.left:.0f} dollars and up) have the largest errors ({band_mae.iloc[-1]:.2f} dollars on average), since the same two "
    f"areas can produce very different routes and traffic.",
    "**Limits.** The data only records community areas, not exact addresses, so trips within or between large areas vary a lot. "
    "Weather is the actual day's weather, standing in for a forecast. The platform (Uber, Lyft, etc.) and ride type are not identified, "
    "and surges often last only minutes [2], shorter than the hourly demand features can see. A production quote engine would add the "
    "exact route distance from a map service.",
]
display(Markdown("\n\n".join(lines)))

## 10. References
1. City of Chicago / Data.gov. *Transportation Network Providers – Trips (2025-)* (fares rounded to the nearest \$2.50; times to the nearest 15 minutes). https://catalog-beta.data.gov/dataset/transportation-network-providers-trips-2025
2. Chen, L., Mislove, A., & Wilson, C. (2015). *Peeking Beneath the Hood of Uber.* ACM Internet Measurement Conference. https://mislove.org/publications/Uber-IMC.pdf
3. scikit-learn documentation. *Histogram-Based Gradient Boosting.* https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting
4. pandas documentation. *Holidays / holiday calendars (`USFederalHolidayCalendar`).* https://pandas.pydata.org/docs/user_guide/timeseries.html#holidays-holiday-calendars
5. scikit-learn documentation. *Permutation feature importance.* https://scikit-learn.org/stable/modules/permutation_importance.html